In [1]:
import os 
os.chdir("..")
from dataclasses import dataclass
import torch
from torch.utils.data import DataLoader
from models.audio_llm import AudioLLMModel, AudioLLMConfig
from transformers import AutoProcessor
from dataset.librispeech import LibriSpeechChatDataset, collate_fn

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

@dataclass
class ModelConfig:
    downsample_factor: int = 10
    lm_model_name: str = "HuggingFaceTB/SmolLM2-360M-Instruct"
    whisper_model_name: str = "openai/whisper-base"
    
    checkpoint_path: str = "outputs/2025-02-03/22-09-39/checkpoints/checkpoint_latest.pt"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

# Create config instance
cfg = ModelConfig()

def initialize_model(cfg):
    try:
        print("Initializing AudioLLM model...")
        
        model_config = AudioLLMConfig(
            downsample_factor=cfg.downsample_factor,
            lm_model_name=cfg.lm_model_name,
            whisper_model_name=cfg.whisper_model_name
        )
        
        model = AudioLLMModel(model_config)
        model.to(cfg.device)
        model.eval()
        torch.set_grad_enabled(False)
        
        
        print("Model initialized successfully")
        
        return model
        
    except Exception as e:
        print(f"Error initializing model: {str(e)}")
        raise

model = initialize_model(cfg)

Initializing AudioLLM model...


You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


Model initialized successfully


In [2]:
def load_checkpoint(model, checkpoint_path):
    try:
        print(f"Loading checkpoint from {checkpoint_path}")
        
        checkpoint = torch.load(checkpoint_path, map_location=cfg.device)
        
        model.projector.load_state_dict(checkpoint['projector_state'])
        
        print(f"Checkpoint loaded successfully (step {checkpoint['global_step']})")
        print(f"Loss at checkpoint: {checkpoint['loss']:.4f}")
        
    except Exception as e:
        print(f"Error loading checkpoint: {str(e)}")
        raise

load_checkpoint(model, cfg.checkpoint_path)

Loading checkpoint from outputs/2025-02-03/22-09-39/checkpoints/checkpoint_latest.pt
Checkpoint loaded successfully (step 39000)
Loss at checkpoint: 0.7494


/tmp/ipykernel_672850/2287412380.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=cfg.device)


In [3]:
def initialize_test_dataset(model):
    """Initialize the LibriSpeech test dataset."""
    try:
        print("Initializing LibriSpeech test dataset...")
        
        # Initialize processor
        processor = AutoProcessor.from_pretrained(cfg.whisper_model_name)
        
        # Create test dataset
        test_dataset = LibriSpeechChatDataset(
            split="test.clean",
            tokenizer=model.tokenizer,
            processor=processor
        )
        
        # Create dataloader
        test_loader = DataLoader(
            test_dataset,
            batch_size=1,
            collate_fn=collate_fn,
            shuffle=True
        )
        
        print("Test dataset initialized successfully")
        print(f"Number of test samples: {len(test_dataset)}")
        
        return test_loader, processor
        
    except Exception as e:
        print(f"Error initializing test dataset: {str(e)}")
        raise

# Initialize test dataset and loader
test_loader, processor = initialize_test_dataset(model)

Initializing LibriSpeech test dataset...
Test dataset initialized successfully
Number of test samples: 2620


In [4]:
def evaluate_model(model, test_loader, num_samples=5):
    """Evaluate the model on LibriSpeech test data."""
    model.eval()
    results = []
    
    try:
        print(f"Starting evaluation on {num_samples} test samples...")
        
        with torch.inference_mode():
            for batch_idx, batch in enumerate(test_loader):
                if batch_idx >= num_samples:
                    break
                    
                input_ids = batch["input_ids"].to(cfg.device)
                audio_features = batch["audio_features"].to(cfg.device, torch.bfloat16)
                labels = batch["labels"].to(cfg.device)
                
                def get_ground_truth(ids):
                    bos_positions = (ids == model.tokenizer.bos_token_id).nonzero(as_tuple=True)[0]
                    eos_positions = (ids == model.tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
                    
                    last_bos = bos_positions[-1].item()
                    
                    eos_after_bos = eos_positions[eos_positions > last_bos]
                    response_end = eos_after_bos[0].item() if len(eos_after_bos) > 0 else len(ids)
                    
                    response_ids = ids[last_bos+4:response_end]
                    return model.tokenizer.decode(response_ids, skip_special_tokens=True)
                
                def get_input(ids):
                    bos_positions = (ids == model.tokenizer.bos_token_id).nonzero(as_tuple=True)[0]
                    eos_positions = (ids == model.tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
                    
                    last_bos = bos_positions[-1].item()
                    
                    eos_after_bos = eos_positions[eos_positions > last_bos]
                    response_end = eos_after_bos[0].item() if len(eos_after_bos) > 0 else len(ids)
                    
                    response_ids = ids[0:last_bos+4]
                    return response_ids
                
                # Get ground truth for each sample in batch
                ground_truths = [get_ground_truth(ids) for ids in input_ids]
                input_ids = [get_input(ids) for ids in input_ids]
                input_ids = torch.stack(input_ids)

                
                # Generate completions
                outputs = model.generate(
                    input_ids=input_ids,
                    audio_features=audio_features,
                    max_new_tokens=128,
                    do_sample=False,
                    # temperature=0.7,     # Lower temperature for more focused sampling
                    # min_p=0.1,
                    # repetition_penalty=1.2,
                    # pad_token_id=model.tokenizer.pad_token_id,
                    # bos_token_id=model.tokenizer.bos_token_id,
                    # eos_token_id=model.tokenizer.eos_token_id,
                    # num_beams=5,
                    # early_stopping=True
                )
                
                
                # Process results for each sample in batch
                for i in range(len(outputs)):
                    # Get the generated text (excluding the prompt)
                    generated_text = model.tokenizer.decode(
                        outputs[i][input_ids[i].size(0):], 
                        skip_special_tokens=False
                    )
                    
                    results.append({
                        'ground_truth': ground_truths[i],
                        'generated': generated_text
                    })
                    
                    print(f"\nSample {len(results)}:")
                    print(f'Input: {model.tokenizer.decode(input_ids[i], skip_special_tokens=False)}')
                    print()
                    print(f"Generated: {generated_text}")
                    print()
                    print(f"Ground truth: {ground_truths[i]}")
                    print()
                    print("-" * 50)
        
        return results
        
    except Exception as e:
        print(f"Error during evaluation: {str(e)}")
        raise
    finally:
        model.train()

# Run evaluation
results = evaluate_model(model, test_loader, 16)

Starting evaluation on 16 test samples...

Sample 1:
Input: <|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Repeat the following: <|audio|><|im_end|>
<|im_start|>assistant


Generated:  of fear and despair which was so full of Satin Pangs of fear and despair which was so full of Satin Pangs of fear and despair which was so full of Satin Pangs of fear and despair which was so full of Satin Pangs of fear and despair which was so full of Satin Pangs of fear and despair which was so full of Satin Pangs of fear and despair which was so full of Satin Pangs of

Ground truth: Still her heart had received a shock in the perception of Anne's delicacy and all these holidays she watched over her with the longing fond anxiety which is so full of sudden pangs of fear

--------------------------------------------------

Sample 2:
Input: <|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<